In [ ]:
# Load master_orders
master_orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/kthdv&dtdm/silver/master_orders"
)

display(master_orders_df.limit(5))

In [ ]:
# Feature Engineering
from pyspark.sql.functions import *
# delivery_time: Số ngày giao hàng thực tế.
master_orders_df = master_orders_df.withColumn(
    "delivery_time",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
)

# estimated_delivery_time
master_orders_df = master_orders_df.withColumn(
    "estimated_delivery_time",
    datediff(
        col("order_estimated_delivery_date"),
        col("order_purchase_timestamp")
    )
)

# delivery_delay: Trễ bao nhiêu ngày
# Dương = giao trễ.
# Âm = giao sớm.
master_orders_df = master_orders_df.withColumn(
    "delivery_delay",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

# shipping_duration: Thời gian từ approve -> carrier
master_orders_df = master_orders_df.withColumn(
    "shipping_duration",
    datediff(
        col("order_delivered_carrier_date"),
        col("order_approved_at")
    )
)

# order_total_value
master_orders_df = master_orders_df.withColumn(
    "order_total_value",
    col("total_product_value") +
    col("total_freight_value")
)

# approval_time 
master_orders_df = master_orders_df.withColumn(
    "approval_time",
    datediff(
        col("order_approved_at"),
        col("order_purchase_timestamp")
    )
)

# Historical customer features available strictly before this purchase.
# Review scores become available at review_answer_timestamp, not purchase time.
from pyspark.sql import Window

order_history_events_df = master_orders_df.select(
    "order_id",
    "customer_unique_id",
    col("order_purchase_timestamp").cast("long").alias("event_time"),
    lit("order").alias("event_type"),
    lit(1).alias("order_increment"),
    col("order_total_value").cast("double").alias("spent_increment"),
    lit(None).cast("double").alias("review_score_increment")
)

review_history_events_df = (
    master_orders_df
    .select(
        lit(None).cast("string").alias("order_id"),
        "customer_unique_id",
        to_timestamp("review_answer_timestamp").cast("long").alias("event_time"),
        col("order_purchase_timestamp").cast("long").alias("review_order_purchase_time"),
        lit("review").alias("event_type"),
        lit(0).alias("order_increment"),
        lit(0.0).alias("spent_increment"),
        col("review_score").cast("double").alias("review_score_increment")
    )
    .filter(col("event_time").isNotNull())
    .filter(col("event_time") > col("review_order_purchase_time"))
    .filter(col("review_score_increment").isNotNull())
    .drop("review_order_purchase_time")
)

history_events_df = order_history_events_df.unionByName(
    review_history_events_df
)

# event_time is numeric so -1 excludes the current event and same-time events.
customer_history_window = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy("event_time")
    .rangeBetween(Window.unboundedPreceding, -1)
)

customer_history_df = (
    history_events_df
    .withColumn(
        "customer_total_orders",
        coalesce(
            sum("order_increment").over(customer_history_window),
            lit(0)
        ).cast("long")
    )
    .withColumn(
        "customer_total_spent",
        coalesce(
            sum("spent_increment").over(customer_history_window),
            lit(0.0)
        )
    )
    .withColumn(
        "avg_review_score_customer",
        avg("review_score_increment").over(customer_history_window)
    )
    .filter(col("event_type") == "order")
    .select(
        "order_id",
        "customer_total_orders",
        "customer_total_spent",
        "avg_review_score_customer"
    )
)

master_orders_df = master_orders_df.join(
    customer_history_df,
    "order_id",
    "left"
)

In [ ]:
# Label Definition

# Binary Classification Label
# Ý nghĩa:
# review_score	| label
# 4-5	        | positive    
# 1-3	        | negative
# Orders without a review do not have a satisfaction label for training.
master_orders_df = (
    master_orders_df
    .filter(col("review_score").isNotNull())
    .withColumn(
        "label",
        when(col("review_score") >= 4, 1).otherwise(0)
    )
)

In [ ]:
# Chose feature for  ML
ml_dataset = master_orders_df.select(
    "order_id",
    "order_purchase_timestamp",
    "payment_type",
    "payment_installments",
    "number_of_items",
    "avg_item_price",
    "delivery_time",
    "delivery_delay",
    "shipping_duration",
    "order_total_value",
    "customer_total_orders",
    "customer_total_spent",
    "avg_review_score_customer",
    "label"
)

In [ ]:
# Handle NULL cho feature

# Check NULL
null_check = ml_dataset.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in ml_dataset.columns
])

display(null_check)

# Fill NULL
ml_dataset = ml_dataset.fillna({
    "payment_type": "unknown",
    "payment_installments": 0,
    "number_of_items": 0,
    "avg_item_price": 0,
    "delivery_time": 0,
    "delivery_delay": 0,
    "shipping_duration": 0,
    "order_total_value": 0,
    "customer_total_orders": 0,
    "customer_total_spent": 0,
    "avg_review_score_customer": 0
})

In [ ]:
# Save Gold Dataset
# This is the last dataset that we will use for training ML model.
ml_dataset.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(
        "/Volumes/workspace/default/kthdv&dtdm/gold/ml_dataset"
    )

In [ ]:
# Validation
ml_rows = ml_dataset.count()
unique_orders = ml_dataset.select("order_id").distinct().count()

print("Rows:", ml_rows)
print("Unique orders:", unique_orders)
print("Columns:", len(ml_dataset.columns))
print("Rows with prior orders:", ml_dataset.filter(col("customer_total_orders") > 0).count())
print("Rows with prior reviews:", ml_dataset.filter(col("avg_review_score_customer") > 0).count())

assert ml_rows == unique_orders, "ml_dataset must contain one row per order_id"

display(ml_dataset.limit(10))